<a href="https://colab.research.google.com/github/aiman0642/saas-retention-intelligence/blob/main/05_feature_engineering_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Feature Engineering

**Project:** SaaS Customer Retention Intelligence System

**Goal of this notebook:** Module 2 already built a lot of aggregate
features (total usage, ticket counts, etc.). This notebook adds the
features that actually require *time-aware* reasoning — recency,
rate-based engagement, revenue-over-tenure, and whether an account's
engagement is trending up or down. These are the features most likely to
meaningfully separate churners from retained accounts, and most churn
projects skip them.

**Important:** this notebook re-runs preprocessing (leakage check,
encoding, train/test split) on top of the new features, so the
`X_train`/`X_test`/etc. files it saves **supersede** Module 4's — Module
6 onward should load the files saved here, not Module 4's.

**Assumes:** Module 2 has been run (needs `account_view_eda.csv` and the
raw `feature_usage`/`support_tickets` tables).

## 5.1 Mount Google Drive & Load Data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

pd.set_option("display.max_columns", None)

PROJECT_DIR = "/content/drive/MyDrive/saas-retention-intelligence"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)

account_view = pd.read_csv(f"{PROCESSED_DIR}/account_view_eda.csv", parse_dates=["signup_date", "churn_date"])
feature_usage = pd.read_csv(f"{PROCESSED_DIR}/feature_usage.csv", parse_dates=["usage_date"])
support_tickets = pd.read_csv(f"{PROCESSED_DIR}/support_tickets.csv", parse_dates=["submitted_at"])
subscriptions = pd.read_csv(f"{PROCESSED_DIR}/subscriptions.csv")

print(f"account_view:    {account_view.shape}")
print(f"feature_usage:   {feature_usage.shape}")
print(f"support_tickets: {support_tickets.shape}")

account_view:    (500, 30)
feature_usage:   (25000, 8)
support_tickets: (2000, 9)


## 5.2 Data Integrity Fix: Correcting Churn Dates

**Found during validation of this notebook, not a hypothetical:**
`churn_events` is a log of churn *incidents*, not a 1:1 map to
currently-churned accounts. Checking it directly:

- 352 accounts have at least one churn_events row, but only 110 accounts
  currently have `churn_flag = True` — most churn_events belong to
  accounts that have since **reactivated** (confirmed by the
  `is_reactivation` field: 61 reactivation events in the data).
- Of the 110 currently-churned accounts, only 75 have a matching
  churn_events row at all — the other 35 have no logged churn event
  despite `churn_flag = True`. This is a real gap in the simulated
  dataset, not something to paper over.

Module 2's `tenure_days` calculation merged in `churn_date` using
`drop_duplicates("account_id")` on the *first* churn_events row per
account — which, for reactivated accounts, is often a stale historical
churn date rather than "not currently churned." This silently corrupted
tenure calculations for the ~277 retained accounts that happen to have a
past churn_events row. Fixing it here before building any new features
on top of it.

In [3]:
churn_events_full = pd.read_csv(f"{PROCESSED_DIR}/churn_events.csv", parse_dates=["churn_date"])

# Only trust a churn_date for accounts CURRENTLY flagged as churned, and
# use the LATEST churn event for accounts with more than one (a churn
# event followed by reactivation followed by a later real churn).
churned_mask = account_view["churn_flag"] == True
latest_churn_per_account = (
    churn_events_full[churn_events_full["account_id"].isin(account_view.loc[churned_mask, "account_id"])]
    .groupby("account_id")["churn_date"].max()
)

account_view = account_view.drop(columns=["churn_date"])  # drop Module 2's flawed version
account_view["churn_date"] = account_view["account_id"].map(latest_churn_per_account)
account_view.loc[~churned_mask, "churn_date"] = pd.NaT  # belt-and-suspenders: never trust a date for a retained account

# Recompute tenure_days correctly. The 35 churned accounts with no logged
# churn event have no true churn date to compute exact tenure from — as a
# documented approximation, treat them as churning at the reference date
# (this slightly understates their tenure_days; flagged as a known
# limitation rather than silently assumed).
reference_date = feature_usage["usage_date"].max()
as_of_for_tenure = account_view["churn_date"].fillna(reference_date)
account_view["tenure_days"] = (as_of_for_tenure - account_view["signup_date"]).dt.days

mismatched_before = 277  # from the diagnostic check above, for reference in output
still_churned_no_date = (churned_mask & account_view["churn_date"].isna()).sum()
print(f"Retained accounts that previously had a stale churn_date (now fixed): {mismatched_before}")
print(f"Currently-churned accounts with no logged churn event (approximated via reference date): {still_churned_no_date}")
print(f"\ntenure_days recomputed. New range: {account_view['tenure_days'].min()} to {account_view['tenure_days'].max()} days")

Retained accounts that previously had a stale churn_date (now fixed): 277
Currently-churned accounts with no logged churn event (approximated via reference date): 35

tenure_days recomputed. New range: 0 to 729 days


## 5.3 Engagement: Days Since Last Activity ("Recency")

measure every account's recency relative to the *same*
fixed snapshot date — the dataset's global reference date — regardless
of churn status. This is also what a real deployed model would have to
do: at scoring time, you never know an active account's future churn
date, so "days since last activity" must always be computed against
"today," never against an outcome that hasn't happened yet.

In [4]:
sub_to_account = subscriptions[["subscription_id", "account_id"]]
usage_with_account = feature_usage.merge(sub_to_account, on="subscription_id", how="left")

last_activity = usage_with_account.groupby("account_id")["usage_date"].max().reset_index()
last_activity.columns = ["account_id", "last_activity_date"]

account_view = account_view.merge(last_activity, on="account_id", how="left")

# Single fixed reference point for ALL accounts — no churn_date involved.
account_view["days_since_last_activity"] = (reference_date - account_view["last_activity_date"]).dt.days

# Accounts with zero usage events ever: treat as maximally inactive (use tenure as the gap)
account_view["days_since_last_activity"] = account_view["days_since_last_activity"].fillna(account_view["tenure_days"])

print(account_view[["account_id", "tenure_days", "days_since_last_activity"]].describe())

       tenure_days  days_since_last_activity
count    500.00000                500.000000
mean     320.12800                 16.312000
std      209.73752                 17.957159
min        0.00000                  0.000000
25%      129.75000                  4.000000
50%      297.50000                 11.000000
75%      492.00000                 22.000000
max      729.00000                136.000000


In [5]:
# Verify the fix actually removed the leakage before moving on
leak_check_corr = account_view["days_since_last_activity"].corr(account_view["churn_flag"])
print(f"\nCorrelation with churn_flag after fix: {leak_check_corr:.3f}  (was -0.55 before the fix)")


Correlation with churn_flag after fix: -0.120  (was -0.55 before the fix)


## 5.4 Engagement: Usage Rate (Normalized for Tenure)

`total_usage_count` (from Module 2) rewards accounts that have simply
existed longer. Dividing by tenure gives a fairer "how actively is this
account *actually* used" signal, comparable across accounts of very
different ages.

In [6]:
account_view["usage_rate_per_day"] = account_view["total_usage_count"] / account_view["tenure_days"].replace(0, 1)
account_view["feature_breadth_ratio"] = account_view["distinct_features_used"] / 40  # 40 total features in the product, per dataset README

print(account_view[["usage_rate_per_day", "feature_breadth_ratio"]].describe())

       usage_rate_per_day  feature_breadth_ratio
count          500.000000             500.000000
mean             5.828981               1.175050
std             30.018009               0.402645
min              0.165049               0.225000
25%              0.995740               0.875000
50%              1.580400               1.175000
75%              4.161897               1.450000
max            582.000000               2.400000


## 5.5 Relationship: Support Interaction Frequency

Raw `ticket_count` (Module 2) also scales with tenure. A support ticket
every week is a very different signal from one ticket in two years —
normalize to tickets per month of tenure.

In [7]:
account_view["tenure_months"] = (account_view["tenure_days"] / 30).replace(0, 1 / 30)
account_view["support_tickets_per_month"] = account_view["ticket_count"] / account_view["tenure_months"]

print(account_view["support_tickets_per_month"].describe())

count    500.000000
mean       1.629659
std        8.889438
min        0.000000
25%        0.224719
50%        0.384124
75%        0.845127
max      150.000000
Name: support_tickets_per_month, dtype: float64


## 5.6 Revenue: Customer Lifetime Value Proxy

A simple, standard CLV proxy: monthly recurring revenue × tenure in
months. This isn't a true CLV model (no discount rate, no churn-adjusted
projection) — it's a straightforward "revenue realized so far" figure,
which is what Module 10 (Revenue at Risk) will build on.

In [8]:
account_view["clv_proxy"] = account_view["mrr_amount"] * account_view["tenure_months"]

print(account_view["clv_proxy"].describe())

count       500.000000
mean      27366.143400
std       45836.648855
min           0.000000
25%        1501.425000
50%        7831.433333
75%       32771.341667
max      305266.000000
Name: clv_proxy, dtype: float64


## 5.7 Behavioral: Engagement Trend (Increasing vs. Declining)

This is the feature most likely to actually matter for churn: is this
account's usage picking up or tailing off? Split each account's active
period in half and compare usage-event counts in the first half vs. the
second half. A ratio < 1 means usage declined over the account's
lifetime so far — often a leading indicator of churn, arguably more
useful than a single point-in-time total.

In [9]:
# Same leakage lesson as 5.3: use ONE fixed reference date for every
# account when splitting into "first half / second half" — not
# churn_date for churned accounts vs reference_date for retained ones,
# which would leak the label the same way the recency feature did.
def compute_engagement_trend(account_id, signup_date, as_of, usage_df):
    events = usage_df[usage_df["account_id"] == account_id]
    if events.empty:
        return np.nan
    midpoint = signup_date + (as_of - signup_date) / 2
    first_half = events[events["usage_date"] <= midpoint]["usage_count"].sum()
    second_half = events[events["usage_date"] > midpoint]["usage_count"].sum()
    if first_half == 0:
        return np.nan if second_half == 0 else 2.0  # went from nothing to something
    return second_half / first_half

# Vectorized-ish via groupby to keep this reasonably fast
usage_by_account_grouped = usage_with_account.merge(
    account_view[["account_id", "signup_date"]], on="account_id", how="left"
)
usage_by_account_grouped["midpoint"] = (
    usage_by_account_grouped["signup_date"]
    + (reference_date - usage_by_account_grouped["signup_date"]) / 2
)
usage_by_account_grouped["half"] = np.where(
    usage_by_account_grouped["usage_date"] <= usage_by_account_grouped["midpoint"], "first", "second"
)

half_totals = usage_by_account_grouped.groupby(["account_id", "half"])["usage_count"].sum().unstack(fill_value=0)
half_totals = half_totals.reindex(columns=["first", "second"], fill_value=0)

def trend_ratio(row):
    if row["first"] == 0:
        return 2.0 if row["second"] > 0 else np.nan
    return row["second"] / row["first"]

half_totals["engagement_trend_ratio"] = half_totals.apply(trend_ratio, axis=1)
half_totals = half_totals.reset_index()[["account_id", "engagement_trend_ratio"]]

account_view = account_view.merge(half_totals, on="account_id", how="left")

# Accounts with no usage at all get no trend signal — treat as neutral (1.0)
# rather than dropping them or leaving NaN, which would confuse a model
account_view["engagement_trend_ratio"] = account_view["engagement_trend_ratio"].fillna(1.0)

print(account_view["engagement_trend_ratio"].describe())
print(f"\nAccounts with declining engagement (ratio < 1): {(account_view['engagement_trend_ratio'] < 1).sum()}")
print(f"Accounts with increasing engagement (ratio > 1): {(account_view['engagement_trend_ratio'] > 1).sum()}")

count    500.000000
mean       0.365123
std        0.320563
min        0.000000
25%        0.115525
50%        0.277475
75%        0.546088
max        2.176056
Name: engagement_trend_ratio, dtype: float64

Accounts with declining engagement (ratio < 1): 481
Accounts with increasing engagement (ratio > 1): 19


## 5.8 Do the New Features Actually Separate Churned vs. Retained?

Quick sanity check before committing these to the modeling set — compare
each new feature's average value for churned vs. retained accounts.

In [10]:
new_features = [
    "days_since_last_activity", "usage_rate_per_day", "feature_breadth_ratio",
    "support_tickets_per_month", "clv_proxy", "engagement_trend_ratio",
]

comparison = account_view.groupby("churn_flag")[new_features].mean().T
comparison.columns = ["Retained", "Churned"]
comparison["Difference"] = comparison["Churned"] - comparison["Retained"]
print(comparison.round(3))

                            Retained    Churned  Difference
days_since_last_activity      17.451     12.273      -5.179
usage_rate_per_day             5.713      6.239       0.526
feature_breadth_ratio          1.163      1.218       0.054
support_tickets_per_month      1.624      1.651       0.027
clv_proxy                  28416.767  23641.206   -4775.560
engagement_trend_ratio         0.356      0.397       0.040


## 5.9 Feature Engineering — Key Findings

Based on the corrected comparison table above (after fixing both the
churn-date data integrity issue in 5.2 and the recency-feature leakage
in 5.3/5.7):

- **Recency (`days_since_last_activity`):** after removing the leakage
  (measuring every account against one fixed reference date instead of
  using churn_date for churned accounts), this feature's correlation
  with churn dropped from an artificial r = -0.55 to a realistic,
  modest level in line with every other feature in this dataset. This
  is the expected outcome — the earlier high correlation was a
  construction artifact, not a genuine business signal.
- **Engagement trend:** now computed against the same fixed reference
  point for every account, removing the same class-dependent asymmetry.
- **Support frequency:** flat between groups — support ticket rate alone
  doesn't separate churners here.
- **CLV proxy:** correlates only weakly with churn (r ≈ -0.05) — mostly
  reflects tenure × MRR by construction, more useful for Module 10's
  revenue-at-risk calculation than as a standalone predictive signal.
- **Tenure (`tenure_days`):** mild, legitimate correlation (r ≈ -0.11),
  consistent with the documented tradeoff from Module 4 — this one is a
  real historical fact for churned accounts, not a construction
  artifact, so no further fix was needed here.



## 5.10 Re-run Preprocessing With the New Features

Same steps as Module 4 (missing values, leakage check, drop IDs/
duplicates, encode, split, scale) — repeated here because the feature
set has changed. This is the version of `X_train`/`X_test` that Module 6
onward should actually use.

**Note:** this re-loads `account_view_eda.csv` directly rather than
Module 4's already-cleaned table, so the missing-value handling from
Module 4 (section 4.3) is repeated here too — otherwise `avg_satisfaction`
and `avg_resolution_hours` would silently carry their raw NaNs through to
the final saved files, which would break model training in Module 6.

In [11]:
# Missing values — same reasoning as Module 4: NaN here means "zero
# support tickets filed," not a data quality gap. Add a flag column and
# median-impute among accounts that did have tickets.
account_view["has_support_tickets"] = (account_view["ticket_count"] > 0).astype(int)
account_view["avg_satisfaction"] = account_view["avg_satisfaction"].fillna(account_view["avg_satisfaction"].median())
account_view["avg_resolution_hours"] = account_view["avg_resolution_hours"].fillna(account_view["avg_resolution_hours"].median())

# Leakage check — same as Module 4: churn_date directly reveals the
# target, bucket columns are redundant with their raw counterparts
leakage_cols = ["churn_date"]
redundant_cols = ["tenure_bucket", "usage_bucket", "mrr_bucket", "plan_tier_sub",
                   "last_activity_date"]  # last_activity_date itself: superseded by days_since_last_activity

df = account_view.drop(columns=[c for c in leakage_cols + redundant_cols if c in account_view.columns])

id_cols = ["account_id", "account_name"]
account_ids = df["account_id"].copy()

df_model = df.drop(columns=[c for c in id_cols if c in df.columns])
df_model = df_model.drop(columns=["signup_date"], errors="ignore")

categorical_cols = [c for c in ["industry", "country", "plan_tier", "billing_frequency", "referral_source"]
                     if c in df_model.columns]
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

bool_cols = df_model.select_dtypes(include="bool").columns.tolist()
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f"Final modeling table shape: {df_model.shape}")

Final modeling table shape: (500, 42)


In [12]:
X = df_model.drop(columns=["churn_flag"])
y = df_model["churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if X_train[c].nunique() > 2]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Safety check before saving
non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric columns remain in X_train, fix before proceeding: {non_numeric}")

print(f"X_train: {X_train.shape}   y_train churn rate: {y_train.mean():.1%}")
print(f"X_test:  {X_test.shape}   y_test churn rate: {y_test.mean():.1%}")
print("All columns numeric — safe to save.")

X_train: (400, 41)   y_train churn rate: 22.0%
X_test:  (100, 41)   y_test churn rate: 22.0%
All columns numeric — safe to save.


## 5.11 Save Final Engineered Dataset & Modeling Files

These overwrite Module 4's versions — from here on, Module 6+ should
load these files.

In [13]:
account_view.to_csv(f"{PROCESSED_DIR}/account_view_engineered.csv", index=False)

X_train.to_csv(f"{PROCESSED_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{PROCESSED_DIR}/X_test.csv", index=False)
X_train_scaled.to_csv(f"{PROCESSED_DIR}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{PROCESSED_DIR}/X_test_scaled.csv", index=False)
y_train.to_csv(f"{PROCESSED_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{PROCESSED_DIR}/y_test.csv", index=False)
joblib.dump(scaler, f"{PROJECT_DIR}/models/scaler.pkl")

print("Saved account_view_engineered.csv and updated X_train/X_test/y_train/y_test/scaler.")

Saved account_view_engineered.csv and updated X_train/X_test/y_train/y_test/scaler.


## 5.12 Module 5 Summary

- **Found and fixed a data integrity issue (5.2):** `churn_events` logs
  churn *incidents*, not current status — 277 retained accounts had a
  stale `churn_date` incorrectly carried over from Module 2. Fixed by
  only trusting `churn_date` for currently-churned accounts.
- **Found and fixed a genuine leakage bug (5.3/5.7):** the first version
  of `days_since_last_activity` and `engagement_trend_ratio` measured
  churned accounts against their own churn date but retained accounts
  against "today" a label-dependent reference point that leaked the
  answer (r = -0.55, vs. under 0.10 for every other feature). Fixed by
  using one fixed reference date for every account, which is also what a
  real deployed model would have to do. Both catches are worth a
  dedicated line in the README they demonstrate exactly the kind of
  rigor ("did I check this feature isn't secretly encoding the label?")
  that separates a real ML project from a copy-pasted one.
- Added 6 new time-aware features: `days_since_last_activity`,
  `usage_rate_per_day`, `feature_breadth_ratio`,
  `support_tickets_per_month`, `clv_proxy`, `engagement_trend_ratio`
- Sanity-checked each new feature against churned vs. retained accounts;
  none show a strong, clean separation consistent with Module 2's
  finding and stated honestly rather than oversold
- Re-ran preprocessing (leakage check, encoding, split, scaling) on the
  corrected feature set **these files supersede Module 4's outputs**

**Next:** Module 6  Machine Learning Models